# Model Evaluation and Visualization

This notebook provides comprehensive evaluation and visualization tools for both detection and segmentation models:

## Features:
- Model performance analysis
- Comprehensive metrics calculation
- Result visualization and comparison
- Error analysis and failure case investigation
- Model interpretability (Grad-CAM, attention maps)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score, jaccard_score
)
import pandas as pd
from tqdm import tqdm
import json
import os
from pathlib import Path
import cv2
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# For model interpretability
from torch.nn import functional as F
from torch import nn
import torch.optim as optim

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Model Loading and Setup

In [ ]:
def load_trained_model(model_path, model_type='detection'):
    """Load trained model from checkpoint"""
    checkpoint = torch.load(model_path, map_location=device)
    config = checkpoint['config']
    
    if model_type == 'detection':
        from detection_model import create_model
        model = create_model(config)
    else:  # segmentation
        from segmentation_model import create_segmentation_model
        model = create_segmentation_model(config)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    
    history = checkpoint.get('history', None)
    
    return model, config, history

class ModelEvaluator:
    """Comprehensive model evaluation framework"""
    
    def __init__(self, detection_model=None, segmentation_model=None):
        self.detection_model = detection_model
        self.segmentation_model = segmentation_model
        
        if detection_model:
            self.detection_model.eval()
        if segmentation_model:
            self.segmentation_model.eval()
    
    def evaluate_detection(self, test_loader, save_results=True):
        """Evaluate detection model"""
        if not self.detection_model:
            raise ValueError("Detection model not provided")
        
        all_predictions = []
        all_probabilities = []
        all_targets = []
        all_features = []
        
        with torch.no_grad():
            for inputs, targets in tqdm(test_loader, desc='Evaluating Detection'):
                inputs = inputs.to(device)
                outputs = self.detection_model(inputs)
                
                probabilities = F.softmax(outputs, dim=1)
                predictions = outputs.argmax(dim=1)
                
                # Extract features for analysis
                if hasattr(self.detection_model, 'extract_features'):
                    features = self.detection_model.extract_features(inputs)
                    all_features.extend(features.cpu().numpy())
                
                all_predictions.extend(predictions.cpu().numpy())
                all_probabilities.extend(probabilities.cpu().numpy())
                all_targets.extend(targets.numpy())
        
        results = {
            'predictions': np.array(all_predictions),
            'probabilities': np.array(all_probabilities),
            'targets': np.array(all_targets),
            'features': np.array(all_features) if all_features else None
        }
        
        if save_results:
            np.savez('detection_results.npz', **results)
            print("Detection results saved to 'detection_results.npz'")
        
        return results
    
    def evaluate_segmentation(self, test_loader, threshold=0.5, save_results=True):
        """Evaluate segmentation model"""
        if not self.segmentation_model:
            raise ValueError("Segmentation model not provided")
        
        all_predictions = []
        all_targets = []
        all_images = []
        all_metrics = []
        
        with torch.no_grad():
            for inputs, targets in tqdm(test_loader, desc='Evaluating Segmentation'):
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = self.segmentation_model(inputs)
                
                predictions = torch.sigmoid(outputs)
                
                # Calculate batch metrics
                batch_metrics = self.calculate_segmentation_metrics(
                    predictions, targets, threshold
                )
                all_metrics.append(batch_metrics)
                
                # Store results
                all_predictions.extend(predictions.cpu().numpy())
                all_targets.extend(targets.cpu().numpy())
                all_images.extend(inputs.cpu().numpy())
        
        # Average metrics
        avg_metrics = {}
        for key in all_metrics[0].keys():
            avg_metrics[key] = np.mean([m[key] for m in all_metrics])
        
        results = {
            'predictions': np.array(all_predictions),
            'targets': np.array(all_targets),
            'images': np.array(all_images),
            'metrics': avg_metrics,
            'batch_metrics': all_metrics
        }
        
        if save_results:
            # Save compressed results (predictions can be large)
            np.savez_compressed('segmentation_results.npz', 
                               predictions=results['predictions'][:10],  # Save only first 10 samples
                               targets=results['targets'][:10],
                               images=results['images'][:10],
                               metrics=results['metrics'])
            print("Segmentation results saved to 'segmentation_results.npz'")
        
        return results
    
    def calculate_segmentation_metrics(self, predictions, targets, threshold=0.5):
        """Calculate segmentation metrics for a batch"""
        predictions_binary = (predictions > threshold).float()
        
        # Flatten for calculation
        pred_flat = predictions_binary.view(-1).cpu().numpy()
        target_flat = targets.view(-1).cpu().numpy()
        
        # Calculate metrics
        intersection = (pred_flat * target_flat).sum()
        union = pred_flat.sum() + target_flat.sum() - intersection
        
        # IoU
        iou = intersection / (union + 1e-8) if union > 0 else 0.0
        
        # Dice Score
        dice = (2 * intersection) / (pred_flat.sum() + target_flat.sum() + 1e-8)
        
        # Pixel Accuracy
        pixel_acc = (pred_flat == target_flat).mean()
        
        # Precision, Recall, F1
        if pred_flat.sum() > 0:
            precision = intersection / pred_flat.sum()
        else:
            precision = 0.0
        
        if target_flat.sum() > 0:
            recall = intersection / target_flat.sum()
        else:
            recall = 0.0
        
        if precision + recall > 0:
            f1 = 2 * (precision * recall) / (precision + recall)
        else:
            f1 = 0.0
        
        return {
            'iou': iou,
            'dice': dice,
            'pixel_accuracy': pixel_acc,
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        }

## 2. Comprehensive Visualization Functions

In [ ]:
class ResultsVisualizer:
    """Comprehensive visualization for model results"""
    
    def __init__(self, figsize=(15, 10)):
        self.figsize = figsize
        plt.style.use('seaborn-v0_8')
    
    def plot_detection_results(self, results):
        """Plot comprehensive detection results"""
        predictions = results['predictions']
        probabilities = results['probabilities']
        targets = results['targets']
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # Confusion Matrix
        cm = confusion_matrix(targets, predictions)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
                   xticklabels=['No Spill', 'Spill'], yticklabels=['No Spill', 'Spill'])
        axes[0, 0].set_title('Confusion Matrix')
        axes[0, 0].set_ylabel('True Label')
        axes[0, 0].set_xlabel('Predicted Label')
        
        # ROC Curve
        fpr, tpr, _ = roc_curve(targets, probabilities[:, 1])
        roc_auc = roc_auc_score(targets, probabilities[:, 1])
        
        axes[0, 1].plot(fpr, tpr, linewidth=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
        axes[0, 1].plot([0, 1], [0, 1], 'k--', linewidth=1)
        axes[0, 1].set_title('ROC Curve')
        axes[0, 1].set_xlabel('False Positive Rate')
        axes[0, 1].set_ylabel('True Positive Rate')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # Precision-Recall Curve
        precision, recall, _ = precision_recall_curve(targets, probabilities[:, 1])
        avg_precision = average_precision_score(targets, probabilities[:, 1])
        
        axes[0, 2].plot(recall, precision, linewidth=2, 
                       label=f'PR Curve (AP = {avg_precision:.3f})')
        axes[0, 2].set_title('Precision-Recall Curve')
        axes[0, 2].set_xlabel('Recall')
        axes[0, 2].set_ylabel('Precision')
        axes[0, 2].legend()
        axes[0, 2].grid(True, alpha=0.3)
        
        # Prediction Distribution
        axes[1, 0].hist(probabilities[targets == 0, 1], bins=30, alpha=0.7, 
                       label='No Spill', density=True)
        axes[1, 0].hist(probabilities[targets == 1, 1], bins=30, alpha=0.7, 
                       label='Spill', density=True)
        axes[1, 0].set_title('Prediction Score Distribution')
        axes[1, 0].set_xlabel('Spill Probability')
        axes[1, 0].set_ylabel('Density')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # Classification Report as Table
        report = classification_report(targets, predictions, output_dict=True,
                                     target_names=['No Spill', 'Spill'])
        report_df = pd.DataFrame(report).iloc[:-1, :].T
        
        axes[1, 1].axis('tight')
        axes[1, 1].axis('off')
        table = axes[1, 1].table(cellText=report_df.round(3).values,
                                rowLabels=report_df.index,
                                colLabels=report_df.columns,
                                cellLoc='center',
                                loc='center')
        axes[1, 1].set_title('Classification Report')
        
        # Feature Distribution (if available)
        if results.get('features') is not None:
            features = results['features']
            # Plot first 2 principal components
            from sklearn.decomposition import PCA
            pca = PCA(n_components=2)
            features_2d = pca.fit_transform(features)
            
            scatter = axes[1, 2].scatter(features_2d[:, 0], features_2d[:, 1], 
                                       c=targets, cmap='viridis', alpha=0.6)
            axes[1, 2].set_title('Feature Space (PCA)')
            axes[1, 2].set_xlabel('First Principal Component')
            axes[1, 2].set_ylabel('Second Principal Component')
            plt.colorbar(scatter, ax=axes[1, 2])
        else:
            axes[1, 2].axis('off')
            axes[1, 2].text(0.5, 0.5, 'Features not available', 
                          ha='center', va='center', transform=axes[1, 2].transAxes)
        
        plt.tight_layout()
        plt.show()
        
        # Print summary statistics
        print("\nDetection Results Summary:")
        print(f"Total samples: {len(targets)}")
        print(f"Accuracy: {(predictions == targets).mean():.4f}")
        print(f"AUC-ROC: {roc_auc:.4f}")
        print(f"Average Precision: {avg_precision:.4f}")
    
    def plot_segmentation_results(self, results, num_samples=8):
        """Plot comprehensive segmentation results"""
        images = results['images'][:num_samples]
        targets = results['targets'][:num_samples]
        predictions = results['predictions'][:num_samples]
        metrics = results['metrics']
        
        # Sample visualization
        fig, axes = plt.subplots(num_samples, 4, figsize=(16, 2 * num_samples))
        if num_samples == 1:
            axes = axes.reshape(1, -1)
        
        for i in range(num_samples):
            # Original image (use first channel if RGB)
            if images[i].shape[0] == 3:  # CHW format
                display_image = images[i][0]  # First channel
            else:
                display_image = images[i].squeeze()
            
            axes[i, 0].imshow(display_image, cmap='gray')
            axes[i, 0].set_title('SAR Image')
            axes[i, 0].axis('off')
            
            # Ground truth
            axes[i, 1].imshow(targets[i].squeeze(), cmap='jet')
            axes[i, 1].set_title('Ground Truth')
            axes[i, 1].axis('off')
            
            # Prediction
            axes[i, 2].imshow(predictions[i].squeeze(), cmap='jet')
            axes[i, 2].set_title('Prediction')
            axes[i, 2].axis('off')
            
            # Overlay
            axes[i, 3].imshow(display_image, cmap='gray')
            axes[i, 3].imshow(predictions[i].squeeze() > 0.5, cmap='jet', alpha=0.5)
            axes[i, 3].set_title('Overlay')
            axes[i, 3].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Metrics visualization
        self.plot_segmentation_metrics(metrics)
    
    def plot_segmentation_metrics(self, metrics):
        """Plot segmentation metrics"""
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        
        # Metrics bar plot
        metric_names = list(metrics.keys())
        metric_values = list(metrics.values())
        
        bars = axes[0].bar(metric_names, metric_values, 
                          color=['skyblue', 'lightgreen', 'lightcoral', 'gold', 'plum', 'lightsalmon'])
        axes[0].set_title('Segmentation Metrics')
        axes[0].set_ylabel('Score')
        axes[0].set_ylim(0, 1.1)
        axes[0].grid(True, alpha=0.3)
        
        # Add value labels on bars
        for bar, value in zip(bars, metric_values):
            axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                        f'{value:.3f}', ha='center', va='bottom')
        
        # Rotate x-axis labels
        axes[0].tick_params(axis='x', rotation=45)
        
        # Metrics radar chart
        angles = np.linspace(0, 2 * np.pi, len(metric_names), endpoint=False).tolist()
        metric_values_radar = metric_values + [metric_values[0]]  # Complete the circle
        angles += [angles[0]]
        
        axes[1] = plt.subplot(122, projection='polar')
        axes[1].plot(angles, metric_values_radar, 'o-', linewidth=2, color='blue')
        axes[1].fill(angles, metric_values_radar, alpha=0.25, color='blue')
        axes[1].set_xticks(angles[:-1])
        axes[1].set_xticklabels(metric_names)
        axes[1].set_ylim(0, 1)
        axes[1].set_title('Metrics Radar Chart')
        axes[1].grid(True)
        
        plt.tight_layout()
        plt.show()
        
        # Print summary
        print("\nSegmentation Results Summary:")
        for metric, value in metrics.items():
            print(f"{metric.capitalize()}: {value:.4f}")
    
    def plot_training_comparison(self, detection_history=None, segmentation_history=None):
        """Compare training histories of both models"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        if detection_history:
            # Detection training curves
            axes[0, 0].plot(detection_history['train_loss'], label='Train Loss', marker='o')
            axes[0, 0].plot(detection_history['val_loss'], label='Val Loss', marker='s')
            axes[0, 0].set_title('Detection Model - Loss')
            axes[0, 0].set_xlabel('Epoch')
            axes[0, 0].set_ylabel('Loss')
            axes[0, 0].legend()
            axes[0, 0].grid(True, alpha=0.3)
            
            axes[0, 1].plot(detection_history['train_acc'], label='Train Acc', marker='o')
            axes[0, 1].plot(detection_history['val_acc'], label='Val Acc', marker='s')
            axes[0, 1].set_title('Detection Model - Accuracy')
            axes[0, 1].set_xlabel('Epoch')
            axes[0, 1].set_ylabel('Accuracy (%)')
            axes[0, 1].legend()
            axes[0, 1].grid(True, alpha=0.3)
        
        if segmentation_history:
            # Segmentation training curves
            axes[1, 0].plot(segmentation_history['train_loss'], label='Train Loss', marker='o')
            axes[1, 0].plot(segmentation_history['val_loss'], label='Val Loss', marker='s')
            axes[1, 0].set_title('Segmentation Model - Loss')
            axes[1, 0].set_xlabel('Epoch')
            axes[1, 0].set_ylabel('Loss')
            axes[1, 0].legend()
            axes[1, 0].grid(True, alpha=0.3)
            
            axes[1, 1].plot(segmentation_history['train_iou'], label='Train IoU', marker='o')
            axes[1, 1].plot(segmentation_history['val_iou'], label='Val IoU', marker='s')
            axes[1, 1].plot(segmentation_history['train_dice'], label='Train Dice', marker='^')
            axes[1, 1].plot(segmentation_history['val_dice'], label='Val Dice', marker='v')
            axes[1, 1].set_title('Segmentation Model - Metrics')
            axes[1, 1].set_xlabel('Epoch')
            axes[1, 1].set_ylabel('Score')
            axes[1, 1].legend()
            axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    def analyze_failure_cases(self, results, model_type='detection', threshold=0.5):
        """Analyze failure cases and common errors"""
        if model_type == 'detection':
            self._analyze_detection_failures(results)
        else:
            self._analyze_segmentation_failures(results, threshold)
    
    def _analyze_detection_failures(self, results):
        """Analyze detection failure cases"""
        predictions = results['predictions']
        probabilities = results['probabilities']
        targets = results['targets']
        
        # Find misclassified samples
        misclassified = predictions != targets
        false_positives = (predictions == 1) & (targets == 0)
        false_negatives = (predictions == 0) & (targets == 1)
        
        print(f"\nFailure Analysis - Detection:")
        print(f"Total misclassified: {misclassified.sum()} / {len(targets)} ({100*misclassified.mean():.1f}%)")
        print(f"False Positives: {false_positives.sum()} ({100*false_positives.mean():.1f}%)")
        print(f"False Negatives: {false_negatives.sum()} ({100*false_negatives.mean():.1f}%)")
        
        # Confidence analysis for failures
        fp_confidence = probabilities[false_positives, 1].mean() if false_positives.any() else 0
        fn_confidence = probabilities[false_negatives, 1].mean() if false_negatives.any() else 0
        
        print(f"\nConfidence Analysis:")
        print(f"False Positive avg confidence: {fp_confidence:.3f}")
        print(f"False Negative avg confidence: {fn_confidence:.3f}")
    
    def _analyze_segmentation_failures(self, results, threshold=0.5):
        """Analyze segmentation failure cases"""
        predictions = results['predictions']
        targets = results['targets']
        batch_metrics = results['batch_metrics']
        
        # Find samples with low IoU scores
        iou_scores = [m['iou'] for m in batch_metrics]
        low_iou_indices = [i for i, iou in enumerate(iou_scores) if iou < 0.3]
        
        print(f"\nFailure Analysis - Segmentation:")
        print(f"Samples with IoU < 0.3: {len(low_iou_indices)} / {len(iou_scores)} ({100*len(low_iou_indices)/len(iou_scores):.1f}%)")
        print(f"Mean IoU: {np.mean(iou_scores):.4f}")
        print(f"Std IoU: {np.std(iou_scores):.4f}")
        print(f"Min IoU: {np.min(iou_scores):.4f}")
        print(f"Max IoU: {np.max(iou_scores):.4f}")

## 3. Model Interpretability - Grad-CAM

In [ ]:
class GradCAM:
    """Gradient-weighted Class Activation Mapping for model interpretability"""
    
    def __init__(self, model, target_layer_name):
        self.model = model
        self.target_layer_name = target_layer_name
        self.gradients = None
        self.activations = None
        
        # Register hooks
        self._register_hooks()
    
    def _register_hooks(self):
        """Register forward and backward hooks"""
        def forward_hook(module, input, output):
            self.activations = output
        
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0]
        
        # Find target layer
        for name, module in self.model.named_modules():
            if name == self.target_layer_name:
                module.register_forward_hook(forward_hook)
                module.register_backward_hook(backward_hook)
                break
    
    def generate_cam(self, input_tensor, class_idx=None):
        """Generate Class Activation Map"""
        # Forward pass
        self.model.eval()
        output = self.model(input_tensor)
        
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()
        
        # Backward pass
        self.model.zero_grad()
        class_score = output[0, class_idx]
        class_score.backward()
        
        # Generate CAM
        gradients = self.gradients[0]  # Remove batch dimension
        activations = self.activations[0]  # Remove batch dimension
        
        # Global average pooling of gradients
        weights = torch.mean(gradients, dim=(1, 2))
        
        # Weighted sum of activation maps
        cam = torch.zeros(activations.shape[1:], dtype=torch.float32)
        for i, w in enumerate(weights):
            cam += w * activations[i]
        
        # Apply ReLU and normalize
        cam = F.relu(cam)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        
        return cam.cpu().numpy(), class_idx
    
    def visualize_cam(self, input_tensor, original_image, class_idx=None, alpha=0.4):
        """Visualize Grad-CAM overlay"""
        cam, predicted_class = self.generate_cam(input_tensor, class_idx)
        
        # Resize CAM to input size
        input_size = original_image.shape[-2:]
        cam_resized = cv2.resize(cam, input_size)
        
        # Create heatmap
        heatmap = plt.cm.jet(cam_resized)
        
        # Prepare original image
        if len(original_image.shape) == 4:  # Batch dimension
            img = original_image[0].permute(1, 2, 0).cpu().numpy()
        else:
            img = original_image.permute(1, 2, 0).cpu().numpy()
        
        # Normalize image for display
        img = (img - img.min()) / (img.max() - img.min())
        
        # Use first channel if grayscale
        if img.shape[2] == 3 and np.allclose(img[:,:,0], img[:,:,1]) and np.allclose(img[:,:,1], img[:,:,2]):
            img = img[:,:,0]
            img = np.stack([img, img, img], axis=2)
        
        # Create overlay
        overlay = alpha * heatmap[:,:,:3] + (1 - alpha) * img
        
        # Plot results
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        # Original image
        if img.shape[2] == 3:
            axes[0].imshow(img[:,:,0], cmap='gray')
        else:
            axes[0].imshow(img)
        axes[0].set_title('Original Image')
        axes[0].axis('off')
        
        # Heatmap
        im = axes[1].imshow(cam_resized, cmap='jet')
        axes[1].set_title(f'Grad-CAM (Class: {predicted_class})')
        axes[1].axis('off')
        plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        
        # Overlay
        axes[2].imshow(overlay)
        axes[2].set_title('Overlay')
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        return cam_resized

def create_interpretability_analysis(model, test_loader, model_type='detection', num_samples=4):
    """Create interpretability analysis for the model"""
    if model_type == 'detection':
        # For detection models, use last conv layer
        target_layer = 'backbone.features'  # Adjust based on your model structure
        
        grad_cam = GradCAM(model, target_layer)
        
        # Get some test samples
        model.eval()
        with torch.no_grad():
            for i, (inputs, targets) in enumerate(test_loader):
                if i >= num_samples:
                    break
                
                inputs = inputs.to(device)
                
                for j in range(min(2, inputs.shape[0])):
                    input_tensor = inputs[j:j+1]
                    original_image = inputs[j]
                    
                    print(f"\nSample {i*2 + j + 1}:")
                    try:
                        cam = grad_cam.visualize_cam(input_tensor, original_image)
                    except Exception as e:
                        print(f"Could not generate Grad-CAM: {e}")
                        print("This might be due to model architecture differences.")
                        break
    else:
        print("Grad-CAM for segmentation models requires different implementation.")
        print("Consider using attention maps or feature visualization instead.")

## 4. Comprehensive Model Comparison

In [ ]:
def create_model_comparison_report(detection_results=None, segmentation_results=None, 
                                  save_report=True):
    """Create comprehensive model comparison report"""
    report = {}
    report_text = []
    
    report_text.append("# Oil Spill Detection and Segmentation Model Evaluation Report\n")
    report_text.append(f"Generated on: {pd.Timestamp.now()}\n")
    
    if detection_results:
        # Detection analysis
        det_predictions = detection_results['predictions']
        det_probabilities = detection_results['probabilities']
        det_targets = detection_results['targets']
        
        # Calculate metrics
        det_accuracy = (det_predictions == det_targets).mean()
        det_auc = roc_auc_score(det_targets, det_probabilities[:, 1])
        det_ap = average_precision_score(det_targets, det_probabilities[:, 1])
        
        # Classification report
        det_report = classification_report(det_targets, det_predictions, 
                                         target_names=['No Spill', 'Spill'], 
                                         output_dict=True)
        
        report['detection'] = {
            'accuracy': det_accuracy,
            'auc_roc': det_auc,
            'average_precision': det_ap,
            'classification_report': det_report
        }
        
        report_text.append("## Detection Model Results\n")
        report_text.append(f"- Accuracy: {det_accuracy:.4f}")
        report_text.append(f"- AUC-ROC: {det_auc:.4f}")
        report_text.append(f"- Average Precision: {det_ap:.4f}")
        report_text.append(f"- Precision (Spill): {det_report['Spill']['precision']:.4f}")
        report_text.append(f"- Recall (Spill): {det_report['Spill']['recall']:.4f}")
        report_text.append(f"- F1-Score (Spill): {det_report['Spill']['f1-score']:.4f}\n")
    
    if segmentation_results:
        # Segmentation analysis
        seg_metrics = segmentation_results['metrics']
        
        report['segmentation'] = seg_metrics
        
        report_text.append("## Segmentation Model Results\n")
        for metric, value in seg_metrics.items():
            report_text.append(f"- {metric.capitalize()}: {value:.4f}")
        report_text.append("\n")
    
    # Model comparison
    if detection_results and segmentation_results:
        report_text.append("## Two-Step Framework Performance\n")
        report_text.append("The two-step approach combines:")
        report_text.append("1. Detection model for initial screening")
        report_text.append("2. Segmentation model for precise boundary delineation\n")
        
        # Theoretical combined performance
        combined_precision = det_report['Spill']['precision'] * seg_metrics['precision']
        combined_recall = det_report['Spill']['recall'] * seg_metrics['recall']
        
        report_text.append(f"Theoretical combined performance:")
        report_text.append(f"- Combined Precision: {combined_precision:.4f}")
        report_text.append(f"- Combined Recall: {combined_recall:.4f}\n")
    
    # Recommendations
    report_text.append("## Recommendations\n")
    
    if detection_results:
        if det_auc > 0.9:
            report_text.append("✅ Detection model shows excellent performance")
        elif det_auc > 0.8:
            report_text.append("⚠️ Detection model shows good performance but could be improved")
        else:
            report_text.append("❌ Detection model needs significant improvement")
    
    if segmentation_results:
        if seg_metrics['iou'] > 0.7:
            report_text.append("✅ Segmentation model shows excellent performance")
        elif seg_metrics['iou'] > 0.5:
            report_text.append("⚠️ Segmentation model shows acceptable performance")
        else:
            report_text.append("❌ Segmentation model needs improvement")
    
    # Save report
    if save_report:
        with open('evaluation_report.md', 'w') as f:
            f.write('\n'.join(report_text))
        
        # Save detailed results
        with open('evaluation_report.json', 'w') as f:
            json.dump(report, f, indent=2, default=str)
        
        print("Evaluation report saved to 'evaluation_report.md' and 'evaluation_report.json'")
    
    # Display report
    print('\n'.join(report_text))
    
    return report

def benchmark_models(models_config):
    """Benchmark multiple model configurations"""
    print("Model Benchmarking")
    print("=" * 50)
    
    results = {}
    
    for config_name, config in models_config.items():
        print(f"\nBenchmarking: {config_name}")
        print("-" * 30)
        
        # This would load and evaluate each model configuration
        # For now, we'll just print the configuration
        for key, value in config.items():
            print(f"{key}: {value}")
        
        # Placeholder for actual benchmarking results
        results[config_name] = {
            'config': config,
            'results': 'Would contain actual evaluation results'
        }
    
    return results

## 5. Demo Evaluation (with Dummy Data)

In [ ]:
# Create dummy evaluation data for demonstration
def create_dummy_evaluation_data():
    """Create dummy data for evaluation demonstration"""
    np.random.seed(42)  # For reproducibility
    
    # Detection results
    n_samples = 1000
    detection_results = {
        'targets': np.random.choice([0, 1], n_samples, p=[0.7, 0.3]),
        'predictions': None,
        'probabilities': None
    }
    
    # Create realistic probabilities based on targets
    probabilities = np.zeros((n_samples, 2))
    for i in range(n_samples):
        if detection_results['targets'][i] == 1:  # Positive sample
            prob_positive = np.random.beta(7, 2)  # Skewed towards high probability
        else:  # Negative sample
            prob_positive = np.random.beta(2, 7)  # Skewed towards low probability
        
        probabilities[i, 1] = prob_positive
        probabilities[i, 0] = 1 - prob_positive
    
    detection_results['probabilities'] = probabilities
    detection_results['predictions'] = probabilities.argmax(axis=1)
    
    # Segmentation results
    n_seg_samples = 100
    segmentation_results = {
        'metrics': {
            'iou': 0.72,
            'dice': 0.84,
            'pixel_accuracy': 0.91,
            'precision': 0.78,
            'recall': 0.86,
            'f1_score': 0.82
        },
        'batch_metrics': [
            {'iou': np.random.beta(8, 3)} for _ in range(n_seg_samples)
        ]
    }
    
    # Create dummy image data
    segmentation_results['images'] = np.random.randn(4, 3, 512, 512)
    segmentation_results['targets'] = np.random.randint(0, 2, (4, 512, 512))
    segmentation_results['predictions'] = np.random.rand(4, 512, 512)
    
    return detection_results, segmentation_results

# Create dummy data and visualizer
print("Creating dummy evaluation data for demonstration...")
det_results, seg_results = create_dummy_evaluation_data()
visualizer = ResultsVisualizer()

print("\n" + "="*60)
print("DETECTION MODEL EVALUATION")
print("="*60)
visualizer.plot_detection_results(det_results)

print("\n" + "="*60)
print("SEGMENTATION MODEL EVALUATION")
print("="*60)
visualizer.plot_segmentation_results(seg_results)

print("\n" + "="*60)
print("FAILURE ANALYSIS")
print("="*60)
visualizer.analyze_failure_cases(det_results, 'detection')
visualizer.analyze_failure_cases(seg_results, 'segmentation')

print("\n" + "="*60)
print("COMPREHENSIVE EVALUATION REPORT")
print("="*60)
report = create_model_comparison_report(det_results, seg_results)

print("\n" + "="*60)
print("EVALUATION COMPLETE")
print("="*60)
print("\nTo use with real models:")
print("1. Load your trained models using load_trained_model()")
print("2. Create ModelEvaluator with your models")
print("3. Run evaluate_detection() and evaluate_segmentation()")
print("4. Use the visualization functions with real results")
print("\nNext: Run the end-to-end pipeline notebook (05_End_to_End_Pipeline.ipynb)")